# SAE Analysis

Sparse Autoencoder feature inspection for all three displacement vectors. Each SAE learns a dictionary of sparse basis directions from a different geometric signal:
- $\delta$ (P − C): Directions of predicted patient-state change
- prediction error (P − T): Directions of systematic prediction failure
- observed trajectory (T − C): Directions of actual patient-state change

**2. Cross-Target Co-Activation Comparison** is important (novel) analysis: it identifies which kinds of real patient dynamics (observed_traj features) the model systematically mispredicts (pred_error features) by measuring co-activation.

### Sections
- **A** — Metadata Correlation (closed vocabulary)
- **B** — Open-Ended Feature Inspection (raw ICD/med enrichment)
- **C** — SAE–Cluster Cross-Reference (topology validation)

In [ ]:
import json
import numpy as np
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
from matplotlib import pyplot as plt

from src.utils.io import EXPERIMENTS_DIR, PROCESSED_DIR, load_metadata
from src.utils.seed import load_seed, set_global_seed
from src.analysis.sae import load_sae, extract_sae_activations, inspect_sae_features, sae_cluster_crossref
from src.analysis.clustering import broadcast_to_samples
from src.analysis.plotting import show_or_savefig

In [ ]:
# -- Config --
MODEL_TAG      = "test_01"
EMB_NAME       = "embeddings_40.npz"
SAE_TARGETS    = ["delta", "pred_error", "observed_traj"]
SEQUENCES_PATH = PROCESSED_DIR / "sequences.jsonl"
SAVE_FIGS      = False
SAVE_DATA      = True

exp_dir  = EXPERIMENTS_DIR / MODEL_TAG
set_global_seed(load_seed(exp_dir))

emb_path = exp_dir / "embeddings" / EMB_NAME
fig_dir  = exp_dir / "sae_analysis" / "figures"

def _sp(name: str):
    return fig_dir / name if SAVE_FIGS else None

In [ ]:
# -- Load embeddings --
npz = np.load(emb_path, allow_pickle=True)
z_context   = npz["z_context"]
z_pred      = npz["z_pred"]
z_target    = npz["z_target"]
labels      = npz["labels"]
subject_ids = npz["subject_ids"]

# -- Compute or compute vectors
vectors = {
    "delta":         npz["delta"]         if "delta"         in npz else z_pred - z_context,
    "pred_error":    npz["pred_error"]    if "pred_error"    in npz else z_pred - z_target,
    "observed_traj": npz["observed_traj"] if "observed_traj" in npz else z_target - z_context,
}
N, D = z_context.shape
print(f"Loaded {emb_path.name}:  N={N}  D={D}")

In [ ]:
# -- Load cluster labels for each target, if existing --
cluster_labels_map: dict[str, np.ndarray | None] = {}
for target in SAE_TARGETS:
    cl_path = exp_dir / f"clusters_{target}" / "cluster_labels.npy"
    if cl_path.exists():
        cluster_labels_map[target] = np.load(cl_path)
        print(f"  Cluster labels loaded for {target}")
    else:
        cluster_labels_map[target] = None
        print(f"  No cluster labels for {target} ({cl_path.name} not found)")

# -- Load metadata and broadcast to sample level --
metadata_patients, feature_names, patient_ids = load_metadata(PROCESSED_DIR)
metadata_samples = broadcast_to_samples(metadata_patients, patient_ids, subject_ids)

In [ ]:
# -- Load SAEs & Extract Activations --

sae_models: dict = {}
sae_activations: dict[str, np.ndarray] = {}

for target in SAE_TARGETS:
    ckpt_path = exp_dir / f"sae_{target}" / "sae_checkpoint.pt"
    if not ckpt_path.exists():
        print(f"  [{target}] checkpoint not found at {ckpt_path} — skipping")
        continue

    model = load_sae(ckpt_path)
    acts = extract_sae_activations(model, vectors[target])

    sae_models[target] = model
    sae_activations[target] = acts

    n_active = int((acts != 0).any(axis=0).sum())
    n_dead = acts.shape[1] - n_active
    print(f"  [{target}] loaded  |  activations {acts.shape}  |  "
          f"{n_active} active features, {n_dead} dead")

loaded_targets = list(sae_activations.keys())
print(f"\nLoaded SAEs: {loaded_targets}")

---

### 1. Feature Inspection per Target

In [ ]:
feature_cards: dict[str, list[dict]] = {}

for target in loaded_targets:
    print(f"\n{'='*60}")
    print(f"  Feature inspection: {target}")
    print(f"{'='*60}")

    cards = inspect_sae_features(
        sae_activations[target],
        subject_ids,
        SEQUENCES_PATH,
        cluster_labels=cluster_labels_map.get(target),
        metadata_features=metadata_samples,
        metadata_feature_names=feature_names,
    )
    feature_cards[target] = cards
    print(f"  {len(cards)} active features inspected")

    for card in cards[:5]:
        idx = card["feature_idx"]
        frac = card["activation_frac"]
        top_icd = card["top_enriched_icd"][:3]
        top_med = card["top_enriched_meds"][:3]
        icd_str = ", ".join(f"{e['code']}(OR={e['odds_ratio']})" for e in top_icd)
        med_str = ", ".join(f"{e['med']}(OR={e['odds_ratio']})" for e in top_med)
        print(f"  F{idx:>3d}  act={frac:.2%}  |  ICD: {icd_str}")
        print(f"{'':>26s}  |  Med: {med_str}")

    if SAVE_DATA:
        out_dir = exp_dir / "sae_analysis"
        out_dir.mkdir(parents=True, exist_ok=True)
        with open(out_dir / f"{target}_feature_cards.json", "w") as f:
            json.dump(cards, f, indent=2, default=str)

### 2. Cross-Target Co-Activation Comparison

Strong co-activation reveals which kinds of patient dynamics (observed_traj directions) the model systematically mispredicts (pred_error directions).

In [ ]:
"""For patients where both pred_error and observed_traj SAEs are loaded, 
compute a co-activation matrix: how often does pred_error feature *i*
fire on the same samples as observed_traj feature *j*?
"""

if "pred_error" not in sae_activations or "observed_traj" not in sae_activations:
    missing = [t for t in ("pred_error", "observed_traj") if t not in sae_activations]
    print(f"Cross-target comparison skipped - missing SAEs: {missing}")
    raise SystemExit()
    
acts_pe = sae_activations["pred_error"]
acts_ot = sae_activations["observed_traj"]

# Binary activation masks
bin_pe = (acts_pe != 0).astype(np.float32)   # (N, F_pe)
bin_ot = (acts_ot != 0).astype(np.float32)   # (N, F_ot)

# Co-activation counts: (F_pe, F_ot) — each entry is the number of
# samples where both features fire
coact = bin_pe.T @ bin_ot  # (F_pe, F_ot)

# Normalise by geometric mean of marginal counts for a Jaccard-like score
marginal_pe = bin_pe.sum(axis=0, keepdims=True).T  # (F_pe, 1)
marginal_ot = bin_ot.sum(axis=0, keepdims=True)    # (1, F_ot)
denom = np.sqrt(marginal_pe * marginal_ot)
denom = np.where(denom > 0, denom, 1.0)
coact_norm = coact / denom

# Filter to active features only
active_pe = np.where(marginal_pe.ravel() > 0)[0]
active_ot = np.where(marginal_ot.ravel() > 0)[0]

In [ ]:
if len(active_pe) > 0 and len(active_ot) > 0:
    from src.analysis.plotting import plot_sae_heatmap_top_features
    
    sub = coact_norm[np.ix_(active_pe, active_ot)]

    # Find top co-activating pairs
    flat_idx = np.argsort(sub.ravel())[::-1][:20]
    rows, cols = np.unravel_index(flat_idx, sub.shape)

    print("Top 20 co-activating pairs (pred_error feat, observed_traj feat, score):")
    for r, c in zip(rows, cols):
        pe_feat = active_pe[r]
        ot_feat = active_ot[c]
        score = sub[r, c]
        n_co = int(coact[pe_feat, ot_feat])
        print(f"  PE_F{pe_feat:<4d} x OT_F{ot_feat:<4d}  "
                f"score={score:.3f}  co-activations={n_co}")

    # Heatmap of top features (limit to top 30 per side by max co-activation)
    plot_sae_heatmap_top_features(sub, active_pe, active_ot, 
                                  save_path=_sp("cross_target_coactivation.png"))
else:
    print("No active features in one or both SAEs — skipping co-activation.")

### 3. SAE–Cluster Cross-Reference

In [ ]:
for target in loaded_targets:
    cl = cluster_labels_map.get(target)
    if cl is None:
        print(f"[{target}] No cluster labels — skipping cross-reference")
        continue

    print(f"\n{'='*60}")
    print(f"  SAE-Cluster cross-reference: {target}")
    print(f"{'='*60}")

    xref = sae_cluster_crossref(sae_activations[target], cl)
    summary = xref["summary"]

    print(f"  Active features:    {summary['n_active_features']}")
    print(f"  Clusters:           {summary['n_clusters']}")
    print(f"  Single-cluster:     {summary['n_features_single_cluster']}  "
          "(SAE feature maps to one HDBSCAN cluster)")
    print(f"  Multi-cluster:      {summary['n_features_multi_cluster']}  "
          "(SAE feature cuts across clusters)")
    print(f"  Multi-feature clusters: {summary['n_multi_feature_clusters']}  "
          "(UMAP cluster contains 3+ SAE features)")
    print()
    print(f"  {summary['interpretation']}")

    if SAVE_DATA:
        out_dir = exp_dir / "sae_analysis"
        out_dir.mkdir(parents=True, exist_ok=True)
        xref_out = {k: v for k, v in xref.items() if k != "heatmap"}
        xref_out["feature_indices"] = xref["feature_indices"]
        with open(out_dir / f"{target}_cluster_crossref.json", "w") as f:
            json.dump(xref_out, f, indent=2, default=str)